# Variational Inference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import emcee
import corner
from scipy.optimize import minimize

torch.set_default_dtype(torch.float64)
%matplotlib inline

Let's set a bimodal mixture of Gaussian as our target distribution 
$$p^*(x) \propto 0.5 \mathcal{N}(x; 1, 6) + 0.5 \mathcal{N}(x;-2, 1)$$

In [ ]:
def log_pstar(x):
    # log p^*(x)
    log_p1 = torch.distributions.Normal(1, 6).log_prob(x)
    log_p2 = torch.distributions.Normal(-2, 1).log_prob(x)
    
    log_mix = torch.stack([log_p1, log_p2], dim=0)
    return torch.logsumexp(log_mix, dim=0) - np.log(2.0)

In [ ]:
_x = torch.tensor(np.linspace(-20, 20, 1000))
_logp = log_pstar(_x)

plt.figure(figsize=(4,3))
plt.plot(_x.numpy(), np.exp(_logp.numpy()), c='k')
plt.xlabel('$x$'); plt.ylabel('$p^*(x)$')
plt.ylim(0, None); plt.show()

Now to implement variation inference we need to 
1. Pick a surrogate distribution $q_\phi(x)$
2. Optimize the Kullback-Leibler divergence $\min\limits_\phi D_{\rm KL}(q_\phi \parallel p)$

For our first example, lets keep it simple and pick a mixture of two Gaussian for $q_\phi$

In [ ]:
def log_qphi(x, mu0, log_sigma0, mu1, log_sigma1): 
    sigma0 = torch.exp(log_sigma0)
    sigma1 = torch.exp(log_sigma1)

    log_p1 = torch.distributions.Normal(mu0, sigma0).log_prob(x)
    log_p2 = torch.distributions.Normal(mu1, sigma1).log_prob(x)
    
    log_mix = torch.stack([log_p1, log_p2], dim=0)
    return torch.logsumexp(log_mix, dim=0) - np.log(2.0)

In this case $\phi = [\mu_0, \log \sigma_0, \mu_1, \log \sigma_1]$

Now lets implement the KL divergence or ELBO: 
$$\min\limits_\phi D_{\rm KL}(q_\phi \parallel p^*) = \min\limits_\phi \int{\rm d}x~q_\phi(x)~\log \frac{q_\phi(x)}{p^*(x)}$$

Recall that we can approximate $D_{\rm KL}$ using MC integration as: 
$$D_{\rm KL}(q_\phi \parallel p^*) \approx \sum\limits_{x'\sim q_\phi(x)} \left[ \log q_\phi(x') - \log p^*(x)\right]$$

In [ ]:
def KL_div(mu0, log_sigma0, mu1, log_sigma1, n_samples=256):
    # Kullback-Leibler divergence
    sigma0 = torch.exp(log_sigma0)
    sigma1 = torch.exp(log_sigma1) 
    
    # reparameterisation trick
    eps = torch.randn(n_samples)
    x = torch.concatenate([mu0+sigma0*eps[:n_samples//2], 
                           mu1+sigma1*eps[n_samples//2:]])
    
    # Monte-Carlo estimate of KL divergence
    log_p = log_pstar(x)
    log_q = log_qphi(x, mu0, log_sigma0, mu1, log_sigma1)

    return (log_q - log_p).mean()

Now let's minimize the KL divergence using the `Adam` optimizer (Kingma & Ba 2014)

In [ ]:
# initialize phi
mu0        = nn.Parameter(torch.tensor([1.0]))
log_sigma0 = nn.Parameter(torch.tensor([0.0]))
mu1        = nn.Parameter(torch.tensor([-1.0]))
log_sigma1 = nn.Parameter(torch.tensor([0.0]))

In [ ]:
optimizer = torch.optim.Adam([mu0, log_sigma0, mu1, log_sigma1], lr=0.05)

In [ ]:
print(r'Epoch       mu0     sigma0    mu1      sigma1      Loss' )
for step in range(1, 501):
    optimizer.zero_grad()
    loss = KL_div(mu0, log_sigma0, mu1, log_sigma1)
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        sigma0 = torch.exp(log_sigma0).item()
        sigma1 = torch.exp(log_sigma1).item()
        print(f"{step:>6}  {mu0.item():>8.4f}  {sigma0:>8.4f} {mu1.item():>8.4f}  {sigma1:>8.4f}  {loss.item():>10.4f}")

Let's check our $q_\phi$ estimate

In [ ]:
_logq = log_qphi(_x, mu0, log_sigma0, mu1, log_sigma1)

plt.figure(figsize=(4,3))
plt.plot(_x.numpy(), np.exp(_logp.numpy()), c='k')
plt.plot(_x.numpy(), np.exp(_logq.detach().numpy()), ls='--')
plt.xlabel('$x$'); plt.ylabel('$p^*(x)$')
plt.ylim(0, None); plt.show()

Excellent agreement. Now we can now use $q_\phi$ to sample $p^*$

In [ ]:
# reparameterisation trick to sample q_phi
eps = torch.randn(1000)
xprime = np.concatenate([
    mu0.item()+np.exp(log_sigma0.item())*eps[:500], 
    mu1.item()+np.exp(log_sigma1.item())*eps[500:]])

plt.figure(figsize=(4,3))
plt.plot(_x.numpy(), np.exp(_logp.numpy()), c='k')
plt.hist(xprime, density=True, range=(-20, 20), bins=40)
plt.xlabel('$x$'); plt.ylabel('$p^*(x)$'); plt.show()

## proposal distribution $q_\phi$
In the previous example, we chose correct functional form. Of course, in practice we often don't know this.

So what happens if we choose poorly?

Let's choose a functional form that we know is incorrect
$$q_\phi = \mathcal{N}(x; \mu, \sigma)$$

In [ ]:
def log_qphi(x, mu, log_sigma): 
    sigma = torch.exp(log_sigma)
    return torch.distributions.Normal(mu, sigma).log_prob(x)

We can reimplement the KL divergence and rerun the optimization for 500 epochs

In [ ]:
def KL_div(mu, log_sigma, n_samples=256):
    # Kullback-Leibler divergence
    sigma = torch.exp(log_sigma)
    
    # reparameterisation trick
    eps = torch.randn(n_samples)
    x = mu + sigma * eps
    
    # Monte-Carlo estimate of KL divergence
    log_p = log_pstar(x)
    log_q = log_qphi(x, mu, log_sigma)

    return (log_q - log_p).mean()

Reinitialize $\phi$ and the Adam optimizer

In [ ]:
mu        = nn.Parameter(torch.tensor([.0]))
log_sigma = nn.Parameter(torch.tensor([0.0]))
optimizer = torch.optim.Adam([mu, log_sigma], lr=0.05)

In [ ]:
print(r'Epoch       mu     sigma      Loss' )
for step in range(1, 501):
    optimizer.zero_grad()
    loss = KL_div(mu, log_sigma)
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        sigma = torch.exp(log_sigma).item()
        print(f"{step:>6}  {mu.item():>8.4f}  {sigma:>8.4f}  {loss.item():>10.4f}")

In [ ]:
_logq = log_qphi(_x, mu, log_sigma)

plt.figure(figsize=(4,3))
plt.plot(_x.numpy(), np.exp(_logp.numpy()), c='k')
plt.plot(_x.numpy(), np.exp(_logq.detach().numpy()), ls='--')
plt.xlabel('$x$'); plt.ylabel('$p^*(x)$')
plt.ylim(0, None); plt.show()

$q_\phi$ does not accurately estimate our target distribution $p^*(x)$

**variation inference is sensitive to choice of $q_\phi$**

# variational inference for Bayesian inference
Now let's try using VI for our favorite Bayesian inference problem of fitting a line to data

In [ ]:
a_true, b_true = 0, 1.75
x = torch.tensor([1,1.5,2,2.5])
y = torch.tensor([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')
plt.xlabel('x'); plt.ylabel('y')

Let's set up our model, likelihood, prior, and posterior

In [ ]:
def model(theta, x):
    a, b = theta.T
    return (a + b * x[:,None]).T

def log_likelihood(theta, x, y, yerr):
    m = model(theta, x)
    return -0.5 * torch.sum((y - m) ** 2 / yerr**2, dim=1)

def log_prior(theta):
    a, b = theta[:, 0], theta[:, 1]
    
    in_bounds = ((a >= -10) & (a <= 10) & (b >= -10) & (b <= 10))

    logp = torch.zeros(theta.shape[0])
    logp[~in_bounds] = -torch.inf
    return logp

def log_posterior(theta, x, y, yerr):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, x, y, yerr)

Now let's pick our proposal distribution --- a 2D multivariate Gaussian.

*quick note: rather than the covariance matrix we're going to provide a lower-triangular Cholesky factor (lower triangular matrix with positive diagonal entries) to ensure that the covariance matrix is postive definite*

In [ ]:
def log_qphi(x, mu, log_diag, offdiag): 
    diag = torch.exp(log_diag)
    L = torch.zeros((2,2))
    L[0,0] = diag[0]
    L[1,1] = diag[1]
    L[1,0] = offdiag
    return torch.distributions.MultivariateNormal(mu, scale_tril=L).log_prob(x)

Next, we can set up the KL divergence similar as before but with the posterior $p(\theta\,|\,X)$ as the target distribution

In [ ]:
def KL_div(mu, log_diag, offdiag, _x, _y, _yerr, n_samples=256):
    diag = torch.exp(log_diag)
    L = torch.zeros((2,2))
    L[0,0] = diag[0]
    L[1,1] = diag[1]
    L[1,0] = offdiag
    
    # sample q_phi 
    _dist = torch.distributions.MultivariateNormal(mu, scale_tril=L)
    thetap = _dist.rsample((n_samples,))
    
    # Monte-Carlo estimate of KL divergence
    log_p = log_posterior(thetap, _x, _y, _yerr)
    log_q = log_qphi(thetap, mu, log_diag, offdiag)

    return (log_q - log_p).mean()

Initialize $\phi$ and optimize

In [ ]:
mu        = nn.Parameter(torch.tensor([0., .0]))
log_diag  = nn.Parameter(torch.tensor([0., 0.]))
offdiag   = nn.Parameter(torch.tensor(0.))
optimizer = torch.optim.Adam([mu, log_diag, offdiag], lr=0.05)

for step in range(1, 1001):
    optimizer.zero_grad()
    loss = KL_div(mu, log_diag, offdiag, x, y, yerr)
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"{step:>6}  {loss.item():>10.4f}")

Let's check the estimated posterior samples from variational inference

In [ ]:
diag = torch.exp(log_diag)
L = torch.tensor([[diag[0], 0.], [offdiag, diag[1]]])
_dist = torch.distributions.MultivariateNormal(mu, scale_tril=L)
samples = _dist.sample((10000,))

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(samples.detach().numpy(), labels=["a", "b"], 
                    truths=[a_true, b_true], fig=fig)

# VI for an Exoplanet Orbit
Let's apply VI on an astronomical problem: the orbit of a hot Jupiter from radial velocities.

A planet on a circular orbit makes its star's line-of-sight velocity oscillate with the orbital period $P$ and semi-amplitude $K$:
$$v(t) = \gamma + a\cos\frac{2\pi t}{P} + b\sin\frac{2\pi t}{P},\qquad K = \sqrt{a^2+b^2},$$

$$\theta = (\gamma, a, b, P)$$

For our "observations" we'll simulate 24 velocities over 45 nights with 6 m/s errors (similar to Mayor & Queloz 1995): $P = 4.231$ d, $K = 55.9$ m/s. 

In [ ]:
rng = np.random.default_rng(51)
P_true, K_true, phase_true, gamma_true, sig_rv = 4.2308, 55.9, 1.1, 12.0, 6.0
t_obs = np.sort(rng.uniform(0, 45, 24))                                    # days
v_obs = gamma_true + K_true * np.cos(2 * np.pi * t_obs / P_true + phase_true) + rng.normal(0, sig_rv, t_obs.size)
theta_true = np.array([gamma_true, K_true * np.cos(phase_true), -K_true * np.sin(phase_true), P_true])

def rv_model(theta, t):
    g, a, b, P = theta
    return g + a * np.cos(2 * np.pi * t / P) + b * np.sin(2 * np.pi * t / P)

plt.figure(figsize=(6, 2.6))
plt.errorbar(t_obs, v_obs, sig_rv, fmt='.k')
plt.xlabel('time [days]'); plt.ylabel('radial velocity [m/s]'); plt.show()

Set up the log posterior using a Gaussian likelihood and a flatp rior with 
`(-200 < g < 200 and -200 < a < 200 and -200 < b < 200 and 1 < P < 10)` 

In [ ]:
def log_posterior(theta):
    return 

## Baseline: `emcee`
For the baseline of our comparison, we'll sample the posterior using `emcee`. 
Initialize it at the MAP... and you know the drill by now :) Be sure to check for convergence. (Use `nwalkers = 32`)

*Careful: the posterior has many local maxima in $P$. When optimizing for MAP, I'd suggest looping over many initial guesses and using the best result.*

## VI 
Use a multi-variate Gaussian surrogate to estimate `log_phat_rv` with variational inference. Use the MAP you used to initialize `emcee` as the starting point. 

In [ ]:
tt, vt = torch.tensor(t_obs), torch.tensor(v_obs)


def log_phat_rv(theta):
    ''' the unnormalized log posterior
    '''
    g, a, b, P = theta.T
    m = g[:, None] + a[:, None] * torch.cos(2 * np.pi * tt / P[:, None]) + b[:, None] * torch.sin(2 * np.pi * tt / P[:, None])
    lp = -0.5 * (((vt - m) / sig_rv) ** 2).sum(1)
    inside = (g.abs() < 200) & (a.abs() < 200) & (b.abs() < 200) & (P > 1) & (P < 10)
    return torch.where(inside, lp, torch.tensor(-1e10))


def q_phi(mu, log_diag, offdiag):
    ''' multi-variate Gaussian surrogate q_phi(theta) = N(mu, L L^T)

    mu       : (d,)           mean
    log_diag : (d,)           log of the diagonal of the Cholesky factor L (keeps it positive)
    offdiag  : (d(d-1)/2,)    entries of L below the diagonal
    returns a torch.distributions.MultivariateNormal: use .rsample() to sample, .log_prob() for log q
    '''
    d = mu.shape[0]
    i, j = torch.tril_indices(d, d, offset=-1)          # positions below the diagonal
    L = torch.diag(torch.exp(log_diag))
    L = L.index_put((i, j), offdiag)                    # fill them in (keeps the autograd graph)
    return torch.distributions.MultivariateNormal(mu, scale_tril=L)

Compare the `emcee` and VI posteriors using `corner`. Overplot the truth parameters.